# 서울 성동구 요식 가맹점 경영 위기 조기 경보 시스템
## 빅콘테스트 2025 - 순위기반 EWS (Rank-based Early Warning System)

**관측기간**: 2023-01 ~ 2024-12 (24개월)  
**핵심 설계 원칙**: 폐업 이진분류 대신 현재 상대적 위치(분위수 순위)를 분기별로 제공

| 레이블 | 정의 | 점포 수 |
|---|---|---|
| `is_closed_obs` | 관측기간(2023-2024) 내 폐업 확인 | 30개 |
| `is_closed_all` | 폐업 기록 전체 (2025년 일괄처리 포함) | 127개 |

In [10]:
# ================================================================
# STEP 1: Setup & Load Raw Data
# ================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = './'
OBS_START = pd.Timestamp('2023-01-01')
OBS_END   = pd.Timestamp('2024-12-31')
REGION    = '성동구'
OUT_PATH  = './p_project_data_v2.csv'
SENTINEL  = -999999.9

df1 = pd.read_csv(f'{DATA_DIR}big_data_set1_f.csv', encoding='cp949')
df2 = pd.read_csv(f'{DATA_DIR}big_data_set2_f.csv', encoding='cp949')
df3 = pd.read_csv(f'{DATA_DIR}big_data_set3_f.csv', encoding='cp949')

print(f'df1 (점포 마스터):  {df1.shape}')
print(f'df2 (월별 매출):    {df2.shape}')
print(f'df3 (월별 고객):    {df3.shape}')
print(f'TA_YM 범위: {df2["TA_YM"].min()} ~ {df2["TA_YM"].max()}')


df1 (점포 마스터):  (4185, 11)
df2 (월별 매출):    (86590, 15)
df3 (월별 고객):    (86590, 17)
TA_YM 범위: 202301 ~ 202412


In [11]:
# ================================================================
# STEP 2: 점포 마스터 구축 (dataset1)
# ================================================================

master = df1[df1['MCT_BSE_AR'].str.contains(REGION, na=False)].copy()
print(f'성동구 점포 수: {len(master)}')

master['ARE_D_dt'] = pd.to_datetime(
    master['ARE_D'].astype(str), format='%Y%m%d', errors='coerce'
)
master['MCT_ME_D_dt'] = master['MCT_ME_D'].apply(
    lambda x: pd.Timestamp(str(int(x))) if pd.notna(x) else pd.NaT
)

# is_closed_obs: 관측기간 내 폐업 (실제 관측된 폐업)
# is_closed_all: 폐업 기록 있는 모든 점포 (2025 일괄처리 포함)
master['is_closed_obs'] = (
    master['MCT_ME_D_dt'].notna() &
    master['MCT_ME_D_dt'].between(OBS_START, OBS_END)
).astype(int)
master['is_closed_all'] = master['MCT_ME_D_dt'].notna().astype(int)

print(f'is_closed_obs (2023-2024 내 폐업): {master["is_closed_obs"].sum()}개')
print(f'is_closed_all (폐업 기록 전체):    {master["is_closed_all"].sum()}개')
print(f'영업중 (폐업 기록 없음):            {master["MCT_ME_D_dt"].isna().sum()}개')

MASTER_COLS = [
    'ENCODED_MCT', 'MCT_NM', 'MCT_BSE_AR',
    'HPSN_MCT_ZCD_NM',
    'HPSN_MCT_BZN_CD_NM',
    'HPSN_MCT_ZCD_NM_1',
    'HPSN_MCT_ZCD_NM_2',
    'ARE_D_dt', 'MCT_ME_D_dt',
    'is_closed_obs', 'is_closed_all',
]
master = master[MASTER_COLS].copy()
print(f'마스터 컬럼: {list(master.columns)}')


성동구 점포 수: 4183
is_closed_obs (2023-2024 내 폐업): 30개
is_closed_all (폐업 기록 전체):    127개
영업중 (폐업 기록 없음):            4056개
마스터 컬럼: ['ENCODED_MCT', 'MCT_NM', 'MCT_BSE_AR', 'HPSN_MCT_ZCD_NM', 'HPSN_MCT_BZN_CD_NM', 'HPSN_MCT_ZCD_NM_1', 'HPSN_MCT_ZCD_NM_2', 'ARE_D_dt', 'MCT_ME_D_dt', 'is_closed_obs', 'is_closed_all']


In [12]:
# ================================================================
# STEP 3: 월별 매출 데이터 정제 (dataset2)
# ================================================================

seongdong_ids = set(master['ENCODED_MCT'])
sales = df2[df2['ENCODED_MCT'].isin(seongdong_ids)].copy()
print(f'매출 패널: {len(sales)}행, {sales["ENCODED_MCT"].nunique()}개 점포')

# 버킷 컬럼 파싱: '5_75-90%' -> 5.0
# 버킷 의미: 1=상위10%이하(최상), 6=하위10%이하(최악)
BUCKET_COLS_D2 = [
    'RC_M1_SAA',       # 매출액 구간
    'RC_M1_TO_UE_CT',  # 이용건수 구간
    'RC_M1_UE_CUS_CN', # 이용고객수 구간
    'RC_M1_AV_NP_AT',  # 건당이용금액 구간
    'MCT_OPE_MS_CN',   # 영업개월수 구간
    'APV_CE_RAT',      # 승인금액비율 등급
]
for col in BUCKET_COLS_D2:
    sales[col] = (
        sales[col].astype(str)
        .str.extract(r'^(\d+)')[0]
        .astype(float)
    )

FLOAT_COLS_D2 = [
    'DLV_SAA_RAT',
    'M1_SME_RY_SAA_RAT',
    'M1_SME_RY_CNT_RAT',
    'M12_SME_RY_SAA_PCE_RT',
    'M12_SME_BZN_SAA_PCE_RT',
    'M12_SME_RY_ME_MCT_RAT',
    'M12_SME_BZN_ME_MCT_RAT',
]
for col in FLOAT_COLS_D2:
    n = (sales[col] == SENTINEL).sum()
    sales[col] = sales[col].replace(SENTINEL, np.nan)
    if n > 0:
        print(f'  [{col}] sentinel {n}개 -> NaN')

print('버킷 컬럼 NaN 수:')
print(sales[BUCKET_COLS_D2].isna().sum().to_string())
print('float 컬럼 NaN 수:')
print(sales[FLOAT_COLS_D2].isna().sum().to_string())


매출 패널: 86231행, 4168개 점포
  [DLV_SAA_RAT] sentinel 57164개 -> NaN
  [M12_SME_BZN_ME_MCT_RAT] sentinel 21264개 -> NaN
버킷 컬럼 NaN 수:
RC_M1_SAA             0
RC_M1_TO_UE_CT        0
RC_M1_UE_CUS_CN       0
RC_M1_AV_NP_AT        0
MCT_OPE_MS_CN         0
APV_CE_RAT         6609
float 컬럼 NaN 수:
DLV_SAA_RAT               57164
M1_SME_RY_SAA_RAT             0
M1_SME_RY_CNT_RAT             0
M12_SME_RY_SAA_PCE_RT         0
M12_SME_BZN_SAA_PCE_RT        0
M12_SME_RY_ME_MCT_RAT         0
M12_SME_BZN_ME_MCT_RAT    21264


In [13]:
# ================================================================
# STEP 4: 월별 고객 데이터 정제 (dataset3)
# ================================================================

cust = df3[df3['ENCODED_MCT'].isin(seongdong_ids)].copy()
print(f'고객 패널: {len(cust)}행, {cust["ENCODED_MCT"].nunique()}개 점포')

# sentinel 교체: int 컬럼은 -999999로 저장될 수 있어 둘 다 처리
NUMERIC_COLS_D3 = [c for c in cust.select_dtypes(include='number').columns if c != 'TA_YM']
SENTINEL_VALS = [SENTINEL, -999999]

for col in NUMERIC_COLS_D3:
    for sv in SENTINEL_VALS:
        try:
            n = int((cust[col] == sv).sum())
        except TypeError:
            continue
        if n > 0:
            cust[col] = cust[col].astype(float).replace(sv, np.nan)
            print(f'  [{col}] sentinel({sv}) {n}개 -> NaN')

print('고객 컬럼 NaN 수:')
print(cust[NUMERIC_COLS_D3].isna().sum().to_string())


고객 패널: 86231행, 4168개 점포
  [M12_MAL_1020_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_MAL_30_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_MAL_40_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_MAL_50_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_MAL_60_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_FME_1020_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_FME_30_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_FME_40_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_FME_50_RAT] sentinel(-999999.9) 2001개 -> NaN
  [M12_FME_60_RAT] sentinel(-999999.9) 2001개 -> NaN
  [MCT_UE_CLN_REU_RAT] sentinel(-999999.9) 1640개 -> NaN
  [MCT_UE_CLN_NEW_RAT] sentinel(-999999.9) 1640개 -> NaN
  [RC_M1_SHC_RSD_UE_CLN_RAT] sentinel(-999999.9) 7304개 -> NaN
  [RC_M1_SHC_WP_UE_CLN_RAT] sentinel(-999999.9) 7304개 -> NaN
  [RC_M1_SHC_FLP_UE_CLN_RAT] sentinel(-999999.9) 7304개 -> NaN
고객 컬럼 NaN 수:
M12_MAL_1020_RAT            2001
M12_MAL_30_RAT              2001
M12_MAL_40_RAT              2001
M12_MAL_50_RAT              2001
M12_MAL_60

In [14]:
# ================================================================
# STEP 5: 패널 병합 + 파생 변수 생성
# ================================================================

monthly = pd.merge(sales, cust, on=['ENCODED_MCT', 'TA_YM'], how='outer')
print(f'monthly (sales+cust): {monthly.shape}')

panel = pd.merge(master, monthly, on='ENCODED_MCT', how='left')
print(f'panel (master+monthly): {panel.shape}')

# TA_YM may be NaN for stores with no monthly records; use apply to avoid float-string "202301.0" parse error
panel['TA_YM_dt'] = pd.to_datetime(
    panel['TA_YM'].apply(lambda x: str(int(x)) if pd.notna(x) else None),
    format='%Y%m', errors='coerce'
)

# tenure_months: 창업일->관측월 경과 개월수 (0 이상으로 clip)
panel['tenure_months'] = (
    (panel['TA_YM_dt'].dt.year  - panel['ARE_D_dt'].dt.year)  * 12 +
    (panel['TA_YM_dt'].dt.month - panel['ARE_D_dt'].dt.month)
).clip(lower=0)

# months_to_close: 관측월 기준 폐업까지 남은 개월수
# 양수=폐업 전, 음수=이미 폐업 후 행, NaN=폐업 기록 없음
def months_diff(end, start):
    if pd.isna(end) or pd.isna(start):
        return np.nan
    return (end.year - start.year) * 12 + (end.month - start.month)

panel['months_to_close'] = [
    months_diff(e, s) for e, s in zip(panel['MCT_ME_D_dt'], panel['TA_YM_dt'])
]
panel['months_to_close'] = panel['months_to_close'].astype(float)

ID_COLS    = ['ENCODED_MCT', 'MCT_NM', 'MCT_BSE_AR',
              'HPSN_MCT_ZCD_NM', 'HPSN_MCT_BZN_CD_NM',
              'HPSN_MCT_ZCD_NM_1', 'HPSN_MCT_ZCD_NM_2']
DATE_COLS  = ['ARE_D_dt', 'MCT_ME_D_dt', 'TA_YM', 'TA_YM_dt']
LABEL_COLS = ['is_closed_obs', 'is_closed_all']
TIME_FEAT  = ['tenure_months', 'months_to_close']

fixed = ID_COLS + DATE_COLS + LABEL_COLS + TIME_FEAT
remaining = [c for c in panel.columns if c not in fixed]
panel = panel[fixed + remaining]

print(f'최종 패널: {panel.shape}')
print(f'컬럼 ({len(panel.columns)}개):')
for i, col in enumerate(panel.columns):
    print(f'  [{i:02d}] {col}')


monthly (sales+cust): (86231, 30)
panel (master+monthly): (86246, 40)
최종 패널: (86246, 43)
컬럼 (43개):
  [00] ENCODED_MCT
  [01] MCT_NM
  [02] MCT_BSE_AR
  [03] HPSN_MCT_ZCD_NM
  [04] HPSN_MCT_BZN_CD_NM
  [05] HPSN_MCT_ZCD_NM_1
  [06] HPSN_MCT_ZCD_NM_2
  [07] ARE_D_dt
  [08] MCT_ME_D_dt
  [09] TA_YM
  [10] TA_YM_dt
  [11] is_closed_obs
  [12] is_closed_all
  [13] tenure_months
  [14] months_to_close
  [15] MCT_OPE_MS_CN
  [16] RC_M1_SAA
  [17] RC_M1_TO_UE_CT
  [18] RC_M1_UE_CUS_CN
  [19] RC_M1_AV_NP_AT
  [20] APV_CE_RAT
  [21] DLV_SAA_RAT
  [22] M1_SME_RY_SAA_RAT
  [23] M1_SME_RY_CNT_RAT
  [24] M12_SME_RY_SAA_PCE_RT
  [25] M12_SME_BZN_SAA_PCE_RT
  [26] M12_SME_RY_ME_MCT_RAT
  [27] M12_SME_BZN_ME_MCT_RAT
  [28] M12_MAL_1020_RAT
  [29] M12_MAL_30_RAT
  [30] M12_MAL_40_RAT
  [31] M12_MAL_50_RAT
  [32] M12_MAL_60_RAT
  [33] M12_FME_1020_RAT
  [34] M12_FME_30_RAT
  [35] M12_FME_40_RAT
  [36] M12_FME_50_RAT
  [37] M12_FME_60_RAT
  [38] MCT_UE_CLN_REU_RAT
  [39] MCT_UE_CLN_NEW_RAT
  [40] RC_M1_SH

In [15]:
# ================================================================
# STEP 6: 검증 리포트 & 저장
# ================================================================
print('=' * 65)
print('검증 리포트')
print('=' * 65)

n_stores = panel['ENCODED_MCT'].nunique()
print(f'[1] 전체 점포 수:         {n_stores}  (기대: ~4168)')

n_obs = panel[panel['is_closed_obs'] == 1]['ENCODED_MCT'].nunique()
n_all = panel[panel['is_closed_all'] == 1]['ENCODED_MCT'].nunique()
print(f'[2] is_closed_obs 점포:   {n_obs}  (기대: 30)')
print(f'    is_closed_all 점포:   {n_all}  (기대: 127)')

print(f'[3] TA_YM 범위: {panel["TA_YM"].min()} ~ {panel["TA_YM"].max()}  (기대: 202301~202412)')

neg_t = (panel['tenure_months'] < 0).sum()
print(f'[4] tenure_months < 0: {neg_t}개  (0이어야 함)')

mtc = panel[panel['is_closed_obs'] == 1]['months_to_close'].dropna()
print(f'[5] months_to_close (is_closed_obs=1): min={mtc.min():.0f}, max={mtc.max():.0f}, mean={mtc.mean():.1f}')

KEY_FEATURES = [
    'RC_M1_SAA', 'RC_M1_TO_UE_CT', 'RC_M1_AV_NP_AT', 'MCT_OPE_MS_CN',
    'DLV_SAA_RAT', 'M12_SME_RY_SAA_PCE_RT', 'M12_SME_BZN_SAA_PCE_RT',
    'MCT_UE_CLN_REU_RAT', 'RC_M1_SHC_FLP_UE_CLN_RAT', 'RC_M1_SHC_RSD_UE_CLN_RAT',
]
print('\n[6] 주요 피처 NaN 비율:')
for col in KEY_FEATURES:
    if col in panel.columns:
        rate = panel[col].isna().mean() * 100
        flag = 'WARNING' if rate > 30 else '      '
        print(f'    {flag} {col:<38}: {rate:5.1f}%')

no_sales = panel[panel['TA_YM'].isna()]['ENCODED_MCT'].nunique()
print(f'\n[7] 매출 기록 없는 점포: {no_sales}개')

panel.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f'\n[SAVED] {OUT_PATH}')
print(f'        {panel.shape[0]}행 x {panel.shape[1]}열')


검증 리포트
[1] 전체 점포 수:         4183  (기대: ~4168)
[2] is_closed_obs 점포:   30  (기대: 30)
    is_closed_all 점포:   127  (기대: 127)
[3] TA_YM 범위: 202301.0 ~ 202412.0  (기대: 202301~202412)
[4] tenure_months < 0: 0개  (0이어야 함)
[5] months_to_close (is_closed_obs=1): min=0, max=22, mean=7.2

[6] 주요 피처 NaN 비율:
           RC_M1_SAA                             :   0.0%
           RC_M1_TO_UE_CT                        :   0.0%
           RC_M1_AV_NP_AT                        :   0.0%
           MCT_OPE_MS_CN                         :   0.0%
    WARNING DLV_SAA_RAT                           :  66.3%
           M12_SME_RY_SAA_PCE_RT                 :   0.0%
           M12_SME_BZN_SAA_PCE_RT                :   0.0%
           MCT_UE_CLN_REU_RAT                    :   1.9%
           RC_M1_SHC_FLP_UE_CLN_RAT              :   8.5%
           RC_M1_SHC_RSD_UE_CLN_RAT              :   8.5%

[7] 매출 기록 없는 점포: 15개

[SAVED] ./p_project_data_v2.csv
        86246행 x 43열
